# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arsal626/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule idea

I will prioritize pages that have enough search visibility to matter and show evidence of either staleness or a CTR opportunity. The baseline is a decision-support ranking for human review, not a prediction that a refresh will definitely improve the page.

The rule will use observable current-window signals only. I will not use `trend_direction`, `trend_pct`, product flags, or any future-window outcome as inputs.

**Reason code:** `refresh_opportunity`

**Action label:** `REVIEW_REFRESH`


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

df.head()


Rows: 30000
Columns: 44

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [19]:
# Signal 1: Staleness

staleness_check = df[
    ["days_since_last_update", "freshness_tier"]
].copy()

print("Missing days_since_last_update:",
      staleness_check["days_since_last_update"].isna().sum())

print("\nFreshness bucket table:")
print(
    staleness_check["freshness_tier"]
    .value_counts(dropna=False)
    .rename_axis("freshness_tier")
    .reset_index(name="n")
)

Missing days_since_last_update: 0

Freshness bucket table:
  freshness_tier      n
0           0-30  20480
1         91-180   9171
2          31-90    175
3           181+    174


In [20]:
staleness_bucket = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_ctr=("ctr", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)

staleness_bucket

,freshness_tier,n,median_impressions,median_ctr,median_position
0,0-30,20480,470.0,0.04,9.9
1,181+,174,15.5,0.00,7.0
2,31-90,175,510.0,0.00,13.9
3,91-180,9171,1692.0,0.10,13.6


Verdict: MIXED

Why? The very stale 181+ group does look weak, with only 15.5 median impressions and 0.00 median CTR, but the relationship isn't consistently monotonic. The 91–180 group actually has the highest median CTR and impressions. So staleness provides some evidence, but staleness alone is not a reliable ranking signal.

In [21]:
# Signal 2: CTR vs position

ctr_position_bucket = (
    df.groupby("position_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

ctr_position_bucket

,position_tier,n,median_ctr,median_impressions
0,deep,1319,0.00,218.0
1,page_1,11814,0.16,1179.5
2,page_3_5,7242,0.03,811.5
3,striking,7304,0.11,874.5
4,top_3,2321,0.00,3.0


Verdict: MIXED

The expected relationship isn't clean. page_1 has the strongest median CTR, while page_3_5 is lower, which supports a CTR/position opportunity. But top_3 having 0.00 median CTR with only 3 median impressions makes that bucket unreliable, and deep is also 0.00.

So we should not claim a simple "lower position = lower CTR" rule.

In [22]:
import os

print(os.path.exists("/content/flyrank-ml-internship"))
print(os.listdir("/content")[:20])

True
['.config', 'flyrank-ml-internship', 'sample_data']


In [23]:
!git clone https://github.com/arsal626/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [24]:
import os

repo_path = "/content/flyrank-ml-internship"

print("Repo exists:", os.path.exists(repo_path))
print("\nRepo contents:")
print(os.listdir(repo_path))

Repo exists: True

Repo contents:
['docs', 'outputs', 'SETUP.md', 'CLAUDE.md', 'LICENSE', 'submission', 'requirements.txt', 'skills', 'AGENTS.md', 'notebooks', '.git', 'GUIDE.md', 'scripts', 'work', '.gitignore', '.github', 'README.md', 'data', 'DATA_USE.md']


In [25]:
import os

for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 2 — Baseline action score

baseline = df.copy()

# Avoid ranking pages with almost no search visibility
baseline["visibility_score"] = np.log1p(baseline["impressions_90d"])

# Normalize staleness so larger values mean more stale
baseline["staleness_score"] = (
    baseline["days_since_last_update"].fillna(0)
    / baseline["days_since_last_update"].fillna(0).max()
)

# Lower CTR = larger opportunity
baseline["low_ctr_score"] = 1 - baseline["ctr"].fillna(baseline["ctr"].median())

# Combine the signals into one transparent score
baseline["score"] = (
    0.45 * baseline["staleness_score"]
    + 0.35 * baseline["low_ctr_score"]
    + 0.20 * (
        baseline["visibility_score"]
        / baseline["visibility_score"].max()
    )
)

# One reason code and one action label
baseline["reason_code"] = "refresh_opportunity"
baseline["action"] = "REVIEW_REFRESH"

# Rank highest priority first
baseline = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action"
    ]
].head(10)


,rank,content_id,client_id,score,reason_code,action
0,1,content_55a5b1c46474,client_4ec9599fc2,0.854472,refresh_opportunity,REVIEW_REFRESH
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.816700,refresh_opportunity,REVIEW_REFRESH
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.815493,refresh_opportunity,REVIEW_REFRESH
3,4,content_6476d1d8c050,client_19581e27de,0.814567,refresh_opportunity,REVIEW_REFRESH
4,5,content_8d56efff1e71,client_4ec9599fc2,0.809330,refresh_opportunity,REVIEW_REFRESH
5,6,content_02b0d6e30129,client_19581e27de,0.806296,refresh_opportunity,REVIEW_REFRESH
6,7,content_e2b702f4f92b,client_4ec9599fc2,0.805148,refresh_opportunity,REVIEW_REFRESH
7,8,content_d25a099b3726,client_19581e27de,0.798728,refresh_opportunity,REVIEW_REFRESH
8,9,content_7a888d3d99c8,client_19581e27de,0.796996,refresh_opportunity,REVIEW_REFRESH
9,10,content_f488400fca67,client_9400f1b21c,0.794724,refresh_opportunity,REVIEW_REFRESH


In [27]:
import os

output_dir = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

print("Output directory ready:", output_dir)

Output directory ready: /content/flyrank-ml-internship/work/outputs


In [28]:
# Write ranked queue

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows written:", len(baseline))

Saved: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 skeptical review

| Rank | Action         | Why it's here                                                                                                                                            | What would make it wrong                                                                                                           |
| ---: | -------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
|    1 | REVIEW_REFRESH | Very stale at 373 days, 35 impressions, and 0.00 CTR; it also ranks on page 1, making it a plausible refresh candidate.                                  | The page may have very low demand, so refreshing it may not produce meaningful gains despite its age.                              |
|    2 | REVIEW_REFRESH | Very stale at 373 days with 0.00 CTR and a page 3–5 position, indicating weak current search performance.                                                | Only 2 impressions means there is almost no evidence of a real CTR opportunity.                                                    |
|    3 | REVIEW_REFRESH | 372 days since update, 0.00 CTR, and a page-1 position suggest a potentially outdated page with poor click performance.                                  | Only 2 impressions makes the observed CTR unreliable.                                                                              |
|    4 | REVIEW_REFRESH | 313 days stale, 304 impressions, 0.00 CTR, and deep ranking make this a stronger refresh candidate because it has meaningful visibility but poor clicks. | The page may be intentionally targeting low-volume or highly specific searches where low CTR is expected.                          |
|    5 | REVIEW_REFRESH | 372 days stale with 0.00 CTR and a page 3–5 position.                                                                                                    | Just 1 impression provides almost no evidence that a refresh is needed.                                                            |
|    6 | REVIEW_REFRESH | 313 days stale, 176 impressions, 0.00 CTR, and page-1 positioning create a reasonable refresh opportunity.                                               | The lack of clicks could be caused by query intent or SERP features rather than outdated content.                                  |
|    7 | REVIEW_REFRESH | 334 days stale, 30 impressions, 0.00 CTR, and page-1 positioning make it worth reviewing.                                                                | Only 30 impressions means the opportunity may be too small to justify refresh work.                                                |
|    8 | REVIEW_REFRESH | 305 days stale, 202 impressions, 0.00 CTR, and a deep position indicate weak visibility and click performance.                                           | Being deep in results may be the main problem, so refreshing content may not improve rankings enough to matter.                    |
|    9 | REVIEW_REFRESH | 313 days stale, 95 impressions, 0.00 CTR, and deep ranking indicate a potentially outdated low-performing page.                                          | Low visibility and deep ranking make it uncertain whether content freshness is the real issue.                                     |
|   10 | REVIEW_REFRESH | 305 days stale, 155 impressions, 0.00 CTR, and page-1 positioning give it meaningful evidence for review.                                                | Low CTR may reflect search intent or competition rather than stale content.                                                        |
|   11 | REVIEW_REFRESH | 334 days stale, page-1 position, and 0.00 CTR indicate a potentially underperforming older page.                                                         | Only 10 impressions makes the CTR signal weak.                                                                                     |
|   12 | REVIEW_REFRESH | 304 days stale, 103 impressions, 0.00 CTR, and page-1 positioning make it a reasonable review candidate.                                                 | The page may simply have low demand, so updating it may not create enough additional traffic.                                      |
|   13 | REVIEW_REFRESH | 304 days stale, 85 impressions, 0.00 CTR, and page-1 positioning indicate possible refresh opportunity.                                                  | 85 impressions is still relatively small, and the low CTR may not be caused by content age.                                        |
|   14 | REVIEW_REFRESH | 301 days stale, 64 impressions, 0.00 CTR, and page 3–5 ranking suggest weak performance from an old page.                                                | The page's ranking position may be the larger issue than freshness.                                                                |
|   15 | REVIEW_REFRESH | 305 days stale, 17 impressions, and 0.00 CTR indicate an old page with weak current engagement.                                                          | Very low impressions make this a weak evidence-based refresh candidate.                                                            |
|   16 | REVIEW_REFRESH | 305 days stale, 15 impressions, 0.00 CTR, and page 3–5 position put it high in the baseline queue.                                                       | The extremely low impression count means the page may not have enough demand to justify intervention.                              |
|   17 | REVIEW_REFRESH | 313 days stale, 7 impressions, 0.00 CTR, and striking position make it appear potentially underperforming.                                               | Only 7 impressions provide almost no reliable evidence of a CTR problem.                                                           |
|   18 | REVIEW_REFRESH | 305 days stale, 13 impressions, 0.00 CTR, and page 3–5 position indicate an old page with little current traffic.                                        | The tiny sample means the observed zero CTR may be random rather than actionable.                                                  |
|   19 | REVIEW_REFRESH | 305 days stale, 13 impressions, 0.00 CTR, and page 3–5 position produce a similar refresh signal.                                                        | Very low impressions make the refresh recommendation uncertain.                                                                    |
|   20 | REVIEW_REFRESH | 304 days stale, page-1 position, and 0.00 CTR place it among the highest-priority stale pages.                                                           | Missing/limited supporting fields and low observed volume make this pick less reliable than a page with stronger traffic evidence. |


In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline.head(20).copy()

top20[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "days_since_last_update",
        "freshness_tier",
        "impressions_90d",
        "ctr",
        "avg_position",
        "position_tier",
        "content_age_days",
        "word_count",
        "content_type",
        "main_intent",
        "reason_code",
        "action"
    ]
]

,rank,content_id,client_id,score,days_since_last_update,freshness_tier,impressions_90d,ctr,avg_position,position_tier,content_age_days,word_count,content_type,main_intent,reason_code,action
0,1,content_55a5b1c46474,client_4ec9599fc2,0.854472,373,181+,35,0.0,7.5,page_1,374,NaN,keyword article,informational,refresh_opportunity,REVIEW_REFRESH
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.816700,373,181+,2,0.0,32.5,page_3_5,373,NaN,keyword article,informational,refresh_opportunity,REVIEW_REFRESH
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.815493,372,181+,2,0.0,7.0,page_1,372,NaN,keyword article,informational,refresh_opportunity,REVIEW_REFRESH
3,4,content_6476d1d8c050,client_19581e27de,0.814567,313,181+,304,0.0,67.8,deep,313,NaN,keyword article,transactional,refresh_opportunity,REVIEW_REFRESH
4,5,content_8d56efff1e71,client_4ec9599fc2,0.809330,372,181+,1,0.0,35.0,page_3_5,372,NaN,keyword article,informational,refresh_opportunity,REVIEW_REFRESH
5,6,content_02b0d6e30129,client_19581e27de,0.806296,313,181+,176,0.0,6.9,page_1,313,NaN,keyword article,transactional,refresh_opportunity,REVIEW_REFRESH
6,7,content_e2b702f4f92b,client_4ec9599fc2,0.805148,334,181+,30,0.0,9.3,page_1,334,1246.0,keyword article,NaN,refresh_opportunity,REVIEW_REFRESH
7,8,content_d25a099b3726,client_19581e27de,0.798728,305,181+,202,0.0,64.5,deep,313,NaN,keyword article,transactional,refresh_opportunity,REVIEW_REFRESH
8,9,content_7a888d3d99c8,client_19581e27de,0.796996,313,181+,95,0.0,67.6,deep,313,NaN,keyword article,transactional,refresh_opportunity,REVIEW_REFRESH
9,10,content_f488400fca67,client_9400f1b21c,0.794724,305,181+,155,0.0,5.7,page_1,306,1113.0,keyword article,NaN,refresh_opportunity,REVIEW_REFRESH


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Identify weak picks in the top 20
top20 = baseline.head(20).copy()

weak_picks = top20[
    top20["impressions_90d"] <= 20
][
    [
        "rank",
        "content_id",
        "score",
        "impressions_90d",
        "ctr",
        "days_since_last_update",
        "avg_position"
    ]
]

print("Weak picks (20 or fewer impressions):")
print(weak_picks.to_string(index=False))


# Leakage check
features_used = [
    "impressions_90d",
    "days_since_last_update",
    "ctr"
]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

found_forbidden = [
    col for col in forbidden_features
    if col in features_used
]

print("\nFeatures used for scoring:")
print(features_used)

print("\nForbidden features found in scoring inputs:")
print(found_forbidden)

assert len(found_forbidden) == 0, "Leakage detected!"

print("\nLeakage check: PASSED")

Weak picks (20 or fewer impressions):
 rank           content_id    score  impressions_90d  ctr  days_since_last_update  avg_position
    2 content_f6fdf87348f6 0.816700                2  0.0                     373          32.5
    3 content_1b4ec72dafd4 0.815493                2  0.0                     372           7.0
    5 content_8d56efff1e71 0.809330                1  0.0                     372          35.0
   11 content_06e19c6486b0 0.789399               10  0.0                     334           5.0
   15 content_f2b4acf220d9 0.761898               17  0.0                     305          17.5
   16 content_afd26a07382d 0.760108               15  0.0                     305          45.1
   17 content_94991fe6268c 0.759223                7  0.0                     313          12.4
   18 content_dd413158df3c 0.758078               13  0.0                     305          46.1
   19 content_026a1e2a82fd 0.758078               13  0.0                     305          34.0
  

### Weak picks

Several top-ranked pages have very few impressions (20 or fewer), which makes their 0% CTR less reliable as evidence of a refresh opportunity. For example, some pages have only 1–2 impressions.

This shows a limitation of the baseline rule: because staleness and low CTR receive substantial weight, very old pages can rank highly even when there is little search evidence. A stronger future version could add a minimum visibility threshold before prioritizing a refresh.

### Leakage check

The scoring rule uses only `impressions_90d`, `days_since_last_update`, and `ctr`. No label-derived fields such as `trend_direction` or `trend_pct`, and no product flags such as `priority_score` or `action_type`, were used.

**Leakage check: PASSED.**

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [31]:
%cd /content/flyrank-ml-internship

!git status

/content/flyrank-ml-internship
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [32]:
import os

notebook_path = "/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb"

print("Notebook exists:", os.path.exists(notebook_path))

if os.path.exists(notebook_path):
    print("Notebook size:", os.path.getsize(notebook_path), "bytes")

Notebook exists: True
Notebook size: 3242 bytes
